In [ ]:
# scripts/parse_sanity_check.py
from pathlib import Path

from edital_rag.ingestion.parsing import parse_document

PDF = Path("data/samples/sample_edital.pdf")

doc = parse_document(PDF)
print(f"páginas: {doc.page_count}, tabelas: {len(doc.tables)}")
print(f"sha256: {doc.sha256[:16]}…, bytes: {doc.size_bytes}")

# 1) Tabela de itens da página 1 tem os dois transformadores?
tabelas_p1 = doc.tables_on_page(1)
assert tabelas_p1, "esperava ao menos uma tabela na p.1"
itens = next(t for t in tabelas_p1 if t.column_index("ITEM") is not None)
codigos = itens.column("CÓDIGO")
print(f"códigos na p.1: {codigos}")
assert "170.001.288" in codigos and "170.001.289" in codigos

# 2) Forward-fill aplicado: especificação deve estar presente nas 2 linhas
esp = itens.column("ESPECIFICAÇÃO") or itens.column("ESPECIFICACAO")
assert all(len(e) > 50 for e in esp if e), "especificação ficou vazia em alguma linha"

# 3) Sem mojibake
assert "ÀÁÂÃÄÅ" not in doc.pages[0].text
assert "Página 58 de 84" not in doc.pages[13].text, "rodapé deveria ter sido removido"

# 4) Hífen quebrado não sobrou
assert "trans-\nformador" not in doc.pages_text
print("OK — parsing válido para o TR de Primavera do Leste")